In [1]:
import clip
import torch

# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model_clip, preprocess = clip.load("ViT-B/32", device=device)

print(f"✅ CLIP loaded!")
print(f"Device: {device}")
print(f"Input resolution: {model_clip.visual.input_resolution}")
print(f"Context length: {model_clip.context_length}")
print(f"Vocab size: {model_clip.vocab_size}")

e:\Sign_Train_First\myenv\Lib\site-packages\clip\clip.py:57: UserWarning: C:\Users\Lenovo/.cache/clip\ViT-B-32.pt exists, but the SHA256 checksum does not match; re-downloading the file
  warnings.warn(f"{download_target} exists, but the SHA256 checksum does not match; re-downloading the file")
100%|███████████████████████████████████████| 338M/338M [03:06<00:00, 1.90MiB/s]


✅ CLIP loaded!
Device: cuda
Input resolution: 224
Context length: 77
Vocab size: 49408


In [2]:
import clip
import torch
import cv2
import numpy as np
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model_clip, preprocess = clip.load("ViT-B/32", device=device)

SCENES = [
    "a hospital room or medical facility",
    "a classroom or school environment",
    "a home living room",
    "an outdoor street or public place",
    "an office or workplace",
    "a kitchen or dining area",
    "a shop or market",
    "a library or study room",
    "a pharmacy or medical shop",
    "a waiting room or reception area",
    "a restaurant or cafe",
    "a community center or gathering place",
]

# Encode text ONCE
text_tokens = clip.tokenize(SCENES).to(device)
with torch.no_grad():
    text_features = model_clip.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("✅ Text features encoded!")
print(f"Text features shape: {text_features.shape}")
print(f"Total scenes: {len(SCENES)}")
for i, s in enumerate(SCENES):
    print(f"  {i:2d}. {s}")

✅ Text features encoded!
Text features shape: torch.Size([12, 512])
Total scenes: 12
   0. a hospital room or medical facility
   1. a classroom or school environment
   2. a home living room
   3. an outdoor street or public place
   4. an office or workplace
   5. a kitchen or dining area
   6. a shop or market
   7. a library or study room
   8. a pharmacy or medical shop
   9. a waiting room or reception area
  10. a restaurant or cafe
  11. a community center or gathering place


In [3]:
# Capture one frame from camera
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

# Convert frame to PIL Image for CLIP
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
pil_image = Image.fromarray(frame_rgb)

# Preprocess for CLIP
image_input = preprocess(pil_image).unsqueeze(0).to(device)

# Encode image
with torch.no_grad():
    image_features = model_clip.encode_image(image_input)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

# Calculate similarity scores
similarity = (image_features @ text_features.T).squeeze(0)
probs      = torch.softmax(similarity * 100, dim=0)

# Print results
print("Scene Classification Results:")
print("-" * 45)
for i, (scene, prob) in enumerate(zip(SCENES, probs)):
    bar   = "█" * int(prob.item() * 50)
    print(f"{i+1:2d}. {scene:40s} {prob.item()*100:5.1f}%  {bar}")

best_idx   = probs.argmax().item()
best_scene = SCENES[best_idx]
best_prob  = probs[best_idx].item() * 100
print(f"\n✅ Detected scene: {best_scene} ({best_prob:.1f}%)")

Scene Classification Results:
---------------------------------------------
 1. a hospital room or medical facility        0.7%  
 2. a classroom or school environment          2.0%  █
 3. a home living room                         0.0%  
 4. an outdoor street or public place          0.0%  
 5. an office or workplace                     2.6%  █
 6. a kitchen or dining area                   0.0%  
 7. a shop or market                           0.1%  
 8. a library or study room                   94.0%  ███████████████████████████████████████████████
 9. a pharmacy or medical shop                 0.3%  
10. a waiting room or reception area           0.2%  
11. a restaurant or cafe                       0.0%  
12. a community center or gathering place      0.1%  

✅ Detected scene: a library or study room (94.0%)
